### NLP GRP 12

### 1. Imports

In [ ]:
import json
import copy
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)

RAW_DIR = Path('data/raw')
CLEAN_DIR = Path('data/clean')
SPLITS = ['train', 'dev', 'test']

### 2. Loading Dataset

In [ ]:
raw = {}
for split in SPLITS:
    with open(RAW_DIR / f'{split}_revised.json', encoding='utf-8') as f:
        raw[split] = json.load(f)

for split, docs in raw.items():
    print(f'{split:<5}: {len(docs):>5} documents')

#### Dataset Structure for Iris' Reference

Each split is a list of documents. Every document is a dictionary with four fields:

| Field | Type | Meaning |
|---|---|---|
| `title` | str | Title of the Wikipedia article, e.g. `'AirAsia Zest'`. |
| `sents` | list of list of str | The document text, already split into sentences and tokens. `sents[i][j]` is token `j` of sentence `i`. |
| `vertexSet` | list of entities | The entities in the document. Each entity is a **list of mentions**: every place in the text that refers to the same real-world thing. An entity's position in this list is its ID. |
| `labels` | list of dict | The relation triples that hold between entities in the document. |

**Mention fields** (each item inside an entity in `vertexSet`):

| Field | Meaning |
|---|---|
| `name` | Text of the mention, e.g. `'Zest Airways, Inc.'`. |
| `sent_id` | Index of the sentence the mention is in. |
| `pos` | `[start, end]` token positions within that sentence. `end` is exclusive, so the tokens are `sents[sent_id][start:end]`. |
| `type` | Entity type: `PER` (person), `ORG` (organisation), `LOC` (location), `TIME`, `NUM` (number or quantity) or `MISC` (anything else, e.g. works and events). |
| `global_pos` | `[g, g]`, where `g` is the start token's position counted across the whole document instead of within the sentence. |
| `index` | ID of the form `'<entity>_<mention>'`, e.g. `'0_2'` is the third mention of entity 0. |

**Label fields** (each item in `labels`):

| Field | Meaning |
|---|---|
| `h` | Index of the **head** entity in `vertexSet`. |
| `t` | Index of the **tail** entity in `vertexSet`. |
| `r` | Relation as a Wikidata property ID, e.g. `P159` = *headquarters location*. There are 96 relation types. |
| `evidence` | Indices of the sentences that support the relation. Can be empty, because Re-DocRED adds many triples without evidence. |

Entity pairs with no label are treated as having no relation (`NA`).

### 3. Data Cleaning

Since Re-DocRED is already human annotated, there is light cleaning involved. Edits made keep token positions and entity indices consistent, ensuring labels point at the correct text.

#### (a) Blank tokens
We check for empty, whitespace (e.g. a non-breaking space `\xa0`)

In [ ]:
def find_blank_tokens(doc):
    hits = []
    for sid, sent in enumerate(doc['sents']):
        for i, tok in enumerate(sent):
            if tok.strip() != '':
                continue
            inside = [m['name'] for ent in doc['vertexSet'] for m in ent
                      if m['sent_id'] == sid and m['pos'][0] <= i < m['pos'][1]]
            context = ' '.join(t if t.strip() else '[BLANK]' for t in sent[max(0, i - 4):i + 5])
            hits.append(dict(title=doc['title'], sent_id=sid, tok_idx=i, token=repr(tok),
                             context=context, in_mention=inside[0] if inside else None))
    return hits

blank_df = pd.DataFrame([dict(split=s, **h) for s in SPLITS for d in raw[s] for h in find_blank_tokens(d)])

print('Blank tokens per split:', blank_df.groupby('split').size().reindex(SPLITS, fill_value=0).to_dict())
print('Documents affected    :', blank_df.groupby('split')['title'].nunique().reindex(SPLITS, fill_value=0).to_dict())
print('Blank token values    :', blank_df['token'].value_counts().to_dict())
print(f"Inside a mention      : {blank_df['in_mention'].notna().mean():.1%}")

print('\nExamples inside a mention:')
display(blank_df[blank_df['in_mention'].notna()].head(3)[['split', 'title', 'in_mention', 'context']])
print('Examples outside a mention:')
display(blank_df[blank_df['in_mention'].isna()].head(3)[['split', 'title', 'context']])

#### (b) Duplicate mentions
We identify entities whose span (same `sent_id` and `pos`) listed more than once.

In [ ]:
def find_duplicate_mentions(doc):
    hits = []
    for ei, ent in enumerate(doc['vertexSet']):
        by_span = defaultdict(list)
        for m in ent:
            by_span[(m['sent_id'], tuple(m['pos']))].append(m['name'])
        for (sid, (s, e)), names in by_span.items():
            for _ in names[1:]:
                hits.append(dict(title=doc['title'], entity=ei, span=f'sent {sid} [{s}:{e}]',
                                 span_text=' '.join(doc['sents'][sid][s:e]),
                                 copies=len(names), names=' | '.join(names),
                                 same_name=len(set(names)) == 1))
    return hits

dup_df = pd.DataFrame([dict(split=s, **h) for s in SPLITS for d in raw[s] for h in find_duplicate_mentions(d)])
dup_spans = dup_df.drop_duplicates(['split', 'title', 'entity', 'span'])  # one row per duplicated span
dup_ents = dup_spans.drop_duplicates(['split', 'title', 'entity'])

print('Extra mention copies per split:', dup_df.groupby('split').size().reindex(SPLITS, fill_value=0).to_dict())
print('Entities affected             :', dup_ents.groupby('split').size().reindex(SPLITS, fill_value=0).to_dict())
print('Copies of a duplicated span   :', dup_spans['copies'].value_counts().sort_index().to_dict())
print(f"Copies with identical name    : {dup_spans['same_name'].mean():.1%}")

print('\nExamples (identical names):')
display(dup_spans[dup_spans['same_name']].head(3)[['split', 'title', 'span', 'copies', 'names']])
print('Examples (names differ, e.g. spacing):')
display(dup_spans[~dup_spans['same_name']].head(3)[['split', 'title', 'span', 'span_text', 'names']])

#### (c) Mixed entity types
To identify entities with multiple relationship labels (by right, reference to same entity should share the same label)

In [ ]:
def find_mixed_type_entities(doc):
    hits = []
    for ei, ent in enumerate(doc['vertexSet']):
        types = Counter(m['type'] for m in ent)
        if len(types) < 2:
            continue
        top = types.most_common()
        hits.append(dict(title=doc['title'], entity=ei, types='/'.join(sorted(types)),
                         type_counts=dict(types), tie=top[0][1] == top[1][1],
                         n_labels=sum(l['h'] == ei or l['t'] == ei for l in doc['labels']),
                         mentions=[(m['name'], m['type']) for m in ent]))
    return hits

mixed_df = pd.DataFrame([dict(split=s, **h) for s in SPLITS for d in raw[s] for h in find_mixed_type_entities(d)])

print('Mixed-type entities per split:', mixed_df.groupby('split').size().reindex(SPLITS, fill_value=0).to_dict())
print(f"No majority type (tie)       : {mixed_df['tie'].mean():.1%}")
print('Relation labels using them   :', mixed_df.groupby('split')['n_labels'].sum().reindex(SPLITS, fill_value=0).to_dict())
print('\nType combinations:')
print(mixed_df['types'].value_counts().to_string())

print('\nExamples:')
display(mixed_df.head(3)[['split', 'title', 'type_counts', 'n_labels', 'mentions']])

#### (d) Overlap between splits
To check if same entity appears more than one dataset

In [ ]:
titles = {s: {d['title'] for d in docs} for s, docs in raw.items()}
for a, b in [('train', 'dev'), ('train', 'test'), ('dev', 'test')]:
    print(f'{a} & {b}:', titles[a] & titles[b] or '-')

#### (e) Apply cleaning

In [ ]:

def remove_blank_tokens(doc, log):
    new_sents, kept_before = [], []
    for sent in doc['sents']:
        kb, new = [0], []
        for tok in sent:
            if tok.strip() != '':
                new.append(tok)
            kb.append(len(new))
        log['blank_tokens_removed'] += len(sent) - len(new)
        new_sents.append(new); kept_before.append(kb)
    assert all(new_sents), f"a sentence in {doc['title']!r} became empty"
    doc['sents'] = new_sents

    sent_offsets = np.cumsum([0] + [len(s) for s in new_sents])
    for ei, ent in enumerate(doc['vertexSet']):
        kept = []
        for m in ent:
            s, e = m['pos']
            kb = kept_before[m['sent_id']]
            m['pos'] = [kb[s], kb[e]]
            if m['pos'][0] == m['pos'][1]:
                log['blank_mentions_dropped'] += 1; continue
            g = int(sent_offsets[m['sent_id']] + m['pos'][0])
            m['global_pos'] = [g, g]
            kept.append(m)
        assert kept, f"entity {ei} in {doc['title']!r} lost all mentions"
        doc['vertexSet'][ei] = kept

In [ ]:
# 2. Normalise mention names
def mention_text(doc, m):
    s, e = m['pos']
    return ' '.join(doc['sents'][m['sent_id']][s:e])

def fix_mention_names(doc, log):
    for ent in doc['vertexSet']:
        for m in ent:
            name = ' '.join(m['name'].split())
            span = mention_text(doc, m)
            if name.replace(' ', '') != span.replace(' ', ''):
                name = span; log['names_fixed'] += 1
            m['name'] = name

In [ ]:
def remove_duplicate_mentions(doc, log):
    for ei, ent in enumerate(doc['vertexSet']):
        seen, kept = set(), []
        for m in ent:
            key = (m['sent_id'], tuple(m['pos']))
            if key in seen:
                log['duplicate_mentions_removed'] += 1; continue
            seen.add(key); kept.append(m)
        for mi, m in enumerate(kept):
            m['index'] = f'{ei}_{mi}'
        doc['vertexSet'][ei] = kept

In [ ]:
def merge_duplicate_labels(doc, log):
    merged = {}
    for l in doc['labels']:
        key = (l['h'], l['t'], l['r'])
        if key in merged:
            merged[key]['evidence'] = sorted(set(merged[key]['evidence']) | set(l['evidence']))
            log['duplicate_labels_merged'] += 1
        else:
            merged[key] = dict(l, evidence=sorted(set(l['evidence'])))
    doc['labels'] = list(merged.values())

In [ ]:
def remove_leaked_docs(train_docs, dev_docs, log):
    dev_titles = {d['title'] for d in dev_docs}
    kept = [d for d in train_docs if d['title'] not in dev_titles]
    log['leaked_docs_removed'] += len(train_docs) - len(kept)
    return kept

In [ ]:
def clean_doc(doc, log):
    doc = copy.deepcopy(doc)
    remove_blank_tokens(doc, log)
    fix_mention_names(doc, log)
    remove_duplicate_mentions(doc, log)
    merge_duplicate_labels(doc, log)
    return doc


clean, cleaning_log = {}, {}
for split in SPLITS:
    log = Counter()
    clean[split] = [clean_doc(d, log) for d in raw[split]]
    cleaning_log[split] = log

clean['train'] = remove_leaked_docs(clean['train'], clean['dev'], cleaning_log['train'])

pd.DataFrame(cleaning_log).fillna(0).astype(int)

### 4. Exploratory Data Analysis

### 5. Saving Cleaned Dataset